# Motorcycle Helmet Detection Using YOLOv8s

This notebook covers the complete workflow for training a YOLOv8s model to detect motorcyclists with and without helmets.
It is designed to run in **Google Colab**.

## 1. Setup and Installation
First, install the necessary libraries.

In [ ]:
!pip install ultralytics roboflow opencv-python-headless gradio

In [ ]:
import torch
from ultralytics import YOLO
from google.colab import drive

# Mount Google Drive to save progress and avoid data loss
drive.mount("/content/drive")

# Create a main project directory in Drive
import os
project_dir = "/content/drive/MyDrive/Helmet_Detection_Project"
os.makedirs(project_dir, exist_ok=True)
print(f"\nProject directory set to: {project_dir}")

# Check GPU availability
print(f"\nSetup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

## 2. Download Dataset from Roboflow
Paste your Roboflow API key and project details below to download the dataset in YOLOv8 format.

In [ ]:
from roboflow import Roboflow
import os

# Move into our Drive project folder so dataset is saved there permanently
os.chdir(project_dir)

# Replace with your own Roboflow API Key, Workspace, and Project Name
rf = Roboflow(api_key="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(1)
dataset = version.download("yolov8")



## 3. Train YOLOv8s Model
Initialize the YOLOv8s pretrained model and start training on the downloaded dataset.

In [ ]:
import time
# Initialize the YOLOv8s model
yolo_model = YOLO('yolov8s.pt')

# ── Hyperparameters ────────────────────────────────────────────────────────────
YOLO_CFG = dict(
    data       = f"{project_dir}/data.yaml",
    epochs     = 80,
    imgsz      = 640,
    batch      = 16,             # safe for ~4-6 GB VRAM on Colab
    device     = 0,              # 0 for GPU
    optimizer  = 'AdamW',
    lr0        = 1e-3,
    lrf        = 0.01,
    momentum   = 0.937,
    weight_decay = 5e-4,
    warmup_epochs = 3,
    # Augmentation (built-in)
    mosaic     = 1.0,
    mixup      = 0.1,
    flipud     = 0.01,
    fliplr     = 0.5,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    scale      = 0.5,
    # Regularisation
    patience   = 20,             # early stopping — avoids overfitting
    save_period= 10,             # checkpoint every 10 epochs
    # Output
    project    = f"{project_dir}/runs",
    name       = 'yolov8s_helmet',
    exist_ok   = True,
    verbose    = True,
    plots      = True,
)

print("Starting YOLOv8s training …")
print(f"Epochs={YOLO_CFG['epochs']}  Batch={YOLO_CFG['batch']}  ImgSz={YOLO_CFG['imgsz']}")
t_start = time.time()

yolo_results = yolo_model.train(**YOLO_CFG)



## 4. Evaluation
Evaluate the model's performance (mAP, Precision, Recall) on the validation set.

In [ ]:
# Validate the model on the validation set
metrics = yolo_model.val()
print(f"mAP50-95: {metrics.box.map}")


## 5. Inference & Testing
Test the trained model on sample images and videos.

In [ ]:
import glob
import os
from IPython.display import Image, display

# Run inference on a test image from the dataset
test_images = glob.glob(f"{project_dir}/test/images/*.jpg")
if test_images:
    sample_image = test_images[0]
    print(f"Running inference on: {sample_image}")
    
    # Save inference results directly to Drive
    res = yolo_model.predict(source=sample_image, conf=0.25, save=True, project=f"{project_dir}/runs", name="predict")
    
    # Display the result
    saved_img_path = os.path.join(res[0].save_dir, os.path.basename(sample_image))
    display(Image(filename=saved_img_path))
else:


In [ ]:
# Run inference on a custom video
# Upload a test video to Colab and change the path below
video_path = "path_to_your_test_video.mp4"

# Uncomment the following lines to run video inference and save directly to Drive
# res_video = yolo_model.predict(source=video_path, conf=0.25, save=True, project=f"{project_dir}/runs", name="predict")


## 6. Save Model to Google Drive
To prevent losing your trained weights when the Colab session disconnects, copy them to your Google Drive.

In [ ]:
print(f"All training runs, weights, and inferences have been automatically and permanently saved to {project_dir}")
print("You will not lose any progress if Google Colab disconnects!")

## 7. Web Demo (Gradio)
Launch a local web interface directly within Colab to test your model interactively.

In [ ]:
import gradio as gr
import cv2
from ultralytics import YOLO

# Load the trained model directly from Google Drive
best_model = YOLO(f"{project_dir}/runs/yolov8s_helmet/weights/best.pt")

def predict_image(img):
    results = best_yolo_model.predict(source=img, conf=0.25)
    res_plotted = results[0].plot()
    return cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)

demo = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="numpy", label="Upload Image"),
    outputs=gr.Image(type="numpy", label="Result"),
    title="Motorcycle Helmet Detection",
    description="Upload an image to detect if riders are wearing helmets using YOLOv8s."
)

# Launch the web demo inline
